# 製造 before/after 產生器（Colab，免 Drive）

從課程 repo 下載自訓練 best.pt（不依賴任何 Drive 帳號）+ Bolts 測試原圖，產出乾淨「原圖→偵測」對照，
最後印出兩個網址（before / after）——把網址貼給老師即可合成上站。Run all。

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
import glob, cv2, subprocess
from getpass import getpass
from roboflow import Roboflow
from ultralytics import YOLO
# 1) 從課程 repo 下載自訓練 best.pt（不綁 Drive 帳號）
!wget -q -O best.pt https://raw.githubusercontent.com/ai-cooperation/yolo-vision-course/main/models/metal_best.pt
# 2) Bolts 測試原圖（貼你的免費 Roboflow key）
rf = Roboflow(api_key=getpass('Roboflow API key: '))
ds = rf.workspace('bolts').project('bolts-final').version(1).download('yolov8')
m = YOLO('best.pt')
imgs = sorted(glob.glob(ds.location + '/test/images/*'))[:8]
best=None; bn=-1; bres=None
for p in imgs:
    r = m.predict(p, conf=0.25, verbose=False)[0]
    if len(r.boxes) > bn: bn=len(r.boxes); best=p; bres=r
cv2.imwrite('before.jpg', cv2.imread(best))
cv2.imwrite('after.jpg', bres.plot())
print('選用偵測最多那張：', bn, '件')
# 3) 上傳（file.io，匿名可用），印出網址貼給老師
import json as _json
for f in ['before.jpg','after.jpg']:
    out = subprocess.run(['curl','-s','-F',f'file=@{f}','https://file.io'],capture_output=True,text=True).stdout
    try: link=_json.loads(out).get('link','')
    except: link=''
    if not link:
        t=subprocess.run(['curl','-s','-F',f'file=@{f}','https://tmpfiles.org/api/v1/upload'],capture_output=True,text=True).stdout
        try: link=_json.loads(t)['data']['url']
        except: link=(out+' | '+t)[:200]
    print(f,'->',link)